<a href="https://colab.research.google.com/github/avikumart/DA-DS-Questions/blob/main/Forage_DS_Sim/TASK_4_modeling_and_insights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# connect with google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# import sklearn modeling functions and eval metrics
import sklearn
from sklearn.model_selection import train_test_split
# randomforest and gradient boosting clf
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# load the finad_df dataset for modeling
finad_df = pd.read_csv('/content/drive/MyDrive/DS job sim - Forage/final_df.csv', index_col=False)
finad_df.head()

,Unnamed: 0,id,forecast_discount_energy,forecast_meter_rent_12m,has_gas,imp_cons,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,churn,price_off_peak_fix,price_mid_peak_fix,off_peak_diff,mid_peak_diff,total_cons,forecast_cons,gross_power,forecast_avg_price
0,0,24011ae4ebbe3035111d65fa7c15bc57,0.0,1.78,t,0.00,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1,40.942265,14.901340,3.700961,-16.226389,18315.333333,0.000,0.0,0.106312
1,1,d29c2c54acc38ff3c0614d0a653813dd,0.0,16.27,f,0.00,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0,44.311375,0.000000,0.177779,0.000000,1553.333333,94.975,0.0,0.072855
2,2,764c75f661154dac3a6c254cd082ea7d,0.0,38.72,f,0.00,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0,44.385450,0.000000,0.177779,0.000000,181.333333,23.980,0.0,0.126847
3,3,bba03439a292a1e166f80264c16191cb,0.0,19.83,f,0.00,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0,44.400265,0.000000,0.177779,0.000000,528.000000,120.020,0.0,0.073347
4,4,149d57cf92fc41cf94415803a877cb4b,0.0,131.73,f,52.32,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0,40.688156,16.275263,0.162916,0.065166,1650.333333,485.875,0.0,0.108457


In [ ]:
# remove the Unnamed: column
final_df = finad_df.drop(columns=['Unnamed: 0'])
final_df.head()

,id,forecast_discount_energy,forecast_meter_rent_12m,has_gas,imp_cons,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,churn,price_off_peak_fix,price_mid_peak_fix,off_peak_diff,mid_peak_diff,total_cons,forecast_cons,gross_power,forecast_avg_price
0,24011ae4ebbe3035111d65fa7c15bc57,0.0,1.78,t,0.00,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1,40.942265,14.901340,3.700961,-16.226389,18315.333333,0.000,0.0,0.106312
1,d29c2c54acc38ff3c0614d0a653813dd,0.0,16.27,f,0.00,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0,44.311375,0.000000,0.177779,0.000000,1553.333333,94.975,0.0,0.072855
2,764c75f661154dac3a6c254cd082ea7d,0.0,38.72,f,0.00,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0,44.385450,0.000000,0.177779,0.000000,181.333333,23.980,0.0,0.126847
3,bba03439a292a1e166f80264c16191cb,0.0,19.83,f,0.00,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0,44.400265,0.000000,0.177779,0.000000,528.000000,120.020,0.0,0.073347
4,149d57cf92fc41cf94415803a877cb4b,0.0,131.73,f,52.32,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0,40.688156,16.275263,0.162916,0.065166,1650.333333,485.875,0.0,0.108457


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Separate target variable 'churn' and drop 'id' from features
X = final_df.drop(columns=['churn', 'id'])
y = final_df['churn']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numerical and categorical columns dynamically
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

# Create a preprocessor using ColumnTransformer
# It will apply StandardScaler to numerical features and OneHotEncoder to categorical features.
# sparse_output=False for OneHotEncoder ensures dense output, which is generally easier for models
# and avoids the sparse matrix centering issue for any potential intermediate steps or models.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # 'passthrough' will keep any columns not specified
                            # 'drop' would remove them. Choosing 'passthrough' for flexibility.
)

# Create the full pipeline with preprocessing and the model
pipeline = Pipeline([
    ('preprocessor', preprocessor), # Apply preprocessing steps
    ('model', RandomForestClassifier(n_estimators=100, max_depth=25, random_state=42))
])

# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Predict on the test data
y_pred = pipeline.predict(X_test)

In [ ]:
# evals on the test data
print(classification_report(y_test, y_pred))
# acc scores
print(accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95      2617
           1       0.92      0.04      0.07       305

    accuracy                           0.90      2922
   macro avg       0.91      0.52      0.51      2922
weighted avg       0.90      0.90      0.86      2922

0.8990417522245038
